
# Brazilian Airline Historical Series Analysis

#### Frederico Horst

### Data Sources:
- Historical air fares by origin, destination and airline: available at [ANAC website](https://sistemas.anac.gov.br/sas/downloads/view/frmDownload.aspx)
- Inflation data, using IPCA index: available at [IBGE website](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos/9256-indice-nacional-de-precos-ao-consumidor-amplo.html?=&t=series-historicas)
- More information on air fares on [ANAC website](https://www.anac.gov.br/assuntos/dados-e-estatisticas/mercado-do-transporte-aereo)

### Goals:
- Build a metrics database: average price, standard deviation and quartils by year and route;
<!-- - Build a database for historical deflated prices.
- Calculate the confidence interval for the average price range by route, considering a 95% confidence.
- Confidence intervals will be calculated by route, not considering airline differences. We want to take a closer look to the consumer point of view. -->


In [1]:
from files_processor import FilesProcessor

df = FilesProcessor().process_files()
df.head()

,Year,Month,Airline,OriginICAO,DestinationICAO,Fare,Seats,YearMonth,OriginCity,Origin,DestinationCity,Destination,Route,RouteAgg
0,2002,1,GLO,SBPA,SBBR,397.0,51,2002-01,Aeroporto Internacional de Porto Alegre / Salg...,POA,Aeroporto Internacional de Brasília / Presiden...,BSB,POA >> BSB,"[POA >> BSB, BSB >> POA]"
1,2002,1,GLO,SBSV,SBRF,272.0,5,2002-01,Aeroporto Internacional de Salvador / Deputado...,SSA,Aeroporto Internacional do Recife/ Guararapes ...,REC,SSA >> REC,"[SSA >> REC, REC >> SSA]"
2,2002,1,GLO,SBFL,SBGL,223.0,196,2002-01,Aeroporto Internacional de Florianópolis / Her...,FLN,Aeroporto Internacional do Rio de Janeiro / Ga...,GIG,FLN >> GIG,"[FLN >> GIG, GIG >> FLN]"
3,2002,1,GLO,SBGL,SBSP,96.0,615,2002-01,Aeroporto Internacional do Rio de Janeiro / Ga...,GIG,Aeroporto Internacional de São Paulo / Congonhas,CGH,GIG >> CGH,"[GIG >> CGH, CGH >> GIG]"
4,2002,1,GLO,SBGL,SBRF,340.0,297,2002-01,Aeroporto Internacional do Rio de Janeiro / Ga...,GIG,Aeroporto Internacional do Recife/ Guararapes ...,REC,GIG >> REC,"[GIG >> REC, REC >> GIG]"


In [2]:
# calculate the weighted average fare, standard deviation and total seats for each route and month
# considering seats as weights for the average fare

import pandas

dfX = df.copy()
# transform the RouteAgg column to string to avoid issues with list comparison during groupby
dfX['RouteAgg'] = dfX['RouteAgg'].apply(lambda x: ' >> '.join(x) if isinstance(x, list) else x)

metrics = dfX.groupby(['RouteAgg', 'Route', 'YearMonth']).apply(lambda x: pandas.Series({
    'WeightedAverageFare': (x['Fare'] * x['Seats']).sum() / x['Seats'].sum(),
    # TODO: weighted vars 
    'FareStdDev': x['Fare'].std(),
    'TotalSeats': x['Seats'].sum()
})).reset_index()



metrics.head(15)

,RouteAgg,Route,YearMonth,WeightedAverageFare,FareStdDev,TotalSeats
0,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-07,517.963648,239.994092,159.0
1,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-08,474.473939,237.175467,165.0
2,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-09,586.108156,266.146863,141.0
3,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-10,496.748429,261.731582,191.0
4,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-11,477.686829,287.018149,205.0
5,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2010-12,316.785982,269.747161,224.0
6,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2012-01,729.279337,384.906113,181.0
7,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2012-02,727.327007,391.515912,147.0
8,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2012-03,425.716070,344.273850,257.0
9,AJU >> BEL >> BEL >> AJU,AJU >> BEL,2012-04,455.820754,368.805584,252.0
